# Qwen3.5-0.8B Continued Pre-Training (CPT) Pipeline
### Kaggle GPU Training Notebook (Phase 1)

This notebook runs full-parameter Continued Pre-Training on **Qwen3.5-0.8B-Base** (~1B target tokens) with Kaggle GPU acceleration (2x T4 or P100):
- 35% Stack v3 Code (`HuggingFaceCode/stack-v3-train`)
- 20% Stack v3 Documentation (`.md`, `.rst`, `README`)
- 20% The Vault (`Fsoft-AIC/the-vault-function`)
- 15% FineWeb-HQ (`epfml/FineWeb-HQ`)
- 10% OpenWebMath (`open-web-math/open-web-math`)

## 1. Hardware & GPU Check

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / (1024**3):.1f} GB)")

## 2. Install Dependencies

In [ ]:
!pip install -q --upgrade transformers datasets accelerate peft datasketch xxhash pyyaml rich huggingface_hub

## 3. Hugging Face Authentication

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✓ Logged in to Hugging Face")
except Exception as e:
    print(f"Manual login needed or secret not found: {e}")

## 4. Verify Base Model Architecture (Text-Only / Strip Vision)

In [ ]:
from transformers import AutoTokenizer, Qwen3_5ForCausalLM

model_id = "Qwen/Qwen3.5-0.8B-Base"
print(f"Loading {model_id} as text-only Qwen3_5ForCausalLM...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = Qwen3_5ForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto"
)

total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Loaded text model with {total_params:,} parameters ({total_params/1e9:.2f}B)")

## 5. Prepare Training Shards

In [ ]:
!mkdir -p /kaggle/working/data
!mkdir -p /kaggle/working/checkpoints
!mkdir -p /kaggle/working/logs

!python scripts/03_process_data.py --output-dir /kaggle/working/data

## 6. Run CPT Training

In [ ]:
!python scripts/05_train_cpt.py --data-dir /kaggle/working/data

## 7. Evaluate Model & Generate Benchmark Report

In [ ]:
!python scripts/06_evaluate.py --compare --base Qwen/Qwen3.5-0.8B-Base --cpt /kaggle/working/checkpoints/final --output /kaggle/working/logs/comparison.json